# Notebook 1: PPE Detection — Getting Started

**Phases 1–2** | ~40 minutes

In this notebook you will:
1. Recap Day 1 concepts and understand today's challenge
2. Verify your environment (GPU, packages, models)
3. Explore the synthetic PPE dataset
4. Run a zero-shot baseline with YOLOe (open-vocabulary detection)
5. Experiment with different prompts — discover what works
6. Auto-label the dataset using a foundation model
7. Inspect the generated labels

> **AI copilot**: You have an AI coding assistant available throughout.
> See `cv_copilot_skill.md` for a system prompt you can paste into
> ChatGPT, Claude, or any AI assistant to turn it into a CV expert
> tuned for this workshop.

---
## 0. Recap from Day 1

Yesterday you saw foundation models in action — detecting objects from nothing
but a text description, no training required. Before we dive into building,
let's lock in the key takeaways:

1. **Foundation models detect concepts from text without training.**
   You type `"hard hat"` and the model finds hard hats. Zero examples needed.

2. **Negation is invisible — the "not" paradox.**
   Models see *pixels*, not logic. You cannot detect "person NOT wearing a helmet"
   because the absence of an object has no visual signature.

3. **Word choice is a first-order lever (prompt sensitivity).**
   The exact words you use in your prompt dramatically change detection results.
   This is just as true for vision models as it is for LLMs.

4. **Golden rule: detect THINGS with models, check RELATIONSHIPS with code.**
   Models are great at finding objects. Logic about how objects relate to each other
   ("is this person wearing that helmet?") belongs in your code, not the model.

---
## The Challenge

**Scenario:** A construction company wants to automate safety compliance monitoring.
Cameras on job sites capture images throughout the day. The safety team needs to
know: *what percentage of workers are non-compliant (not wearing a hardhat)?*

**Your task across these three notebooks:**

| Notebook | Goal |
|----------|------|
| **NB01** (this one) | Establish a baseline, auto-label the dataset |
| **NB02** | Inspect labels, curate data, train a fast detector |
| **NB03** | Evaluate, build the compliance system, benchmark speed |

By the end, you will have:
- A detector that runs at **30+ FPS**
- A method to assess **per-image compliance** (% of workers without hardhats)

How you derive compliance from raw detections is **your design challenge**.
The notebooks will give you the building blocks — the architecture is yours to figure out.

---
## 1. Environment Check

Quick verification that GPU and packages are available.
The thorough check was in NB00 — this is just a sanity check.

In [ ]:
import sys, torch, platform
from pathlib import Path

print(f"Python:       {sys.version}")
print(f"PyTorch:      {torch.__version__}")
print(f"Platform:     {platform.system()} {platform.machine()}")

# Auto-detect device: CUDA > MPS > CPU
if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"CUDA:         True ({torch.version.cuda})")
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    print(f"MPS:          True (Apple Silicon)")
else:
    DEVICE = "cpu"
    print(f"CUDA:         False")
    print(f"MPS:          False")

print(f"Device:       {DEVICE}")

import ultralytics
print(f"Ultralytics:  {ultralytics.__version__}")

import transformers
print(f"Transformers: {transformers.__version__}")

# Set paths relative to this notebook
WORKSHOP_DIR = Path(".").resolve().parent
DATA_DIR = WORKSHOP_DIR / "data"
IMAGES_DIR = DATA_DIR / "synthetic_ppe"

print(f"\nWorkshop dir: {WORKSHOP_DIR}")
print(f"Data dir:     {DATA_DIR}")
assert DATA_DIR.exists(), f"Data directory not found: {DATA_DIR}"
print("\n\u2713 Environment OK")


---
## 2. Dataset Exploration

Our dataset: **91 synthetic construction site images** generated with Gemini,
spread across 11 scene categories:

| Category | Count | Description |
|----------|-------|-------------|
| `ambiguous` | 5 | Deliberately hard cases |
| `baseline` | 5 | Standard construction scenes |
| `cctv` | 10 | Security camera angles |
| `challenging` | 5 | Difficult detection scenarios |
| `close_up` | 12 | Close-up 2–4 workers |
| `easy` | 5 | Unambiguous, simple scenes |
| `edge_cases` | 10 | Challenging lighting/occlusion |
| `highrise` | 10 | High-rise building sites |
| `highway` | 9 | Road construction |
| `mixed_compliance` | 10 | Mixed hard hat compliance |
| `warehouse` | 10 | Indoor warehouse scenes |

Let's look at them.

In [ ]:
# ── Utilities: image grid, label counting, label visualization ─────────────

import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import random


def show_image_grid(image_paths, ncols=4, figsize=(16, 12), title=None):
    """Display a grid of images with their filenames."""
    nrows = (len(image_paths) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    if nrows == 1:
        axes = [axes]
    axes = [ax for row in axes for ax in (row if hasattr(row, '__iter__') else [row])]

    for ax, path in zip(axes, image_paths):
        img = Image.open(path)
        ax.imshow(img)
        ax.set_title(Path(path).parent.name + "/" + Path(path).stem, fontsize=8)
        ax.axis("off")
    for ax in axes[len(image_paths):]:
        ax.axis("off")
    if title:
        fig.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()


def count_labels(dataset_dir):
    """Count labels per class in a YOLO dataset directory."""
    import yaml
    dataset_dir = Path(dataset_dir)

    yaml_path = dataset_dir / "dataset.yaml"
    class_names = {}
    if yaml_path.exists():
        with open(yaml_path) as f:
            cfg = yaml.safe_load(f)
        class_names = cfg.get("names", {})

    counts = {}
    for split in ["train", "val"]:
        labels_dir = dataset_dir / "labels" / split
        if not labels_dir.exists():
            continue
        split_counts = {}
        for lf in sorted(labels_dir.glob("*.txt")):
            for line in lf.read_text().strip().splitlines():
                cls_id = int(line.split()[0])
                name = class_names.get(cls_id, f"class_{cls_id}")
                split_counts[name] = split_counts.get(name, 0) + 1
        counts[split] = split_counts

    return counts


CLASS_COLORS = {
    0: (0, 200, 0),      # hardhat — green
    1: (220, 30, 30),     # person — red
    2: (30, 100, 220),    # person (3-class) — blue
}


def show_labeled_image(image_path, label_path, class_names=None, figsize=(12, 8)):
    """Display an image with YOLO-format bounding box labels overlaid."""
    img = np.array(Image.open(image_path))
    h, w = img.shape[:2]

    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(img)

    if Path(label_path).exists():
        for line in Path(label_path).read_text().strip().splitlines():
            parts = line.split()
            cls_id = int(parts[0])
            cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])

            x1 = int((cx - bw / 2) * w)
            y1 = int((cy - bh / 2) * h)
            x2 = int((cx + bw / 2) * w)
            y2 = int((cy + bh / 2) * h)

            color = np.array(CLASS_COLORS.get(cls_id, (200, 200, 200))) / 255
            name = class_names.get(cls_id, f"class_{cls_id}") if class_names else f"class_{cls_id}"

            rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                 linewidth=2, edgecolor=color, facecolor="none")
            ax.add_patch(rect)
            ax.text(x1, y1 - 4, name, color=color, fontsize=8, fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.2", facecolor="black", alpha=0.7))

    ax.set_title(Path(image_path).name)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


print("Utilities loaded.")

In [ ]:
# Inventory: count images per category
categories = sorted([d for d in IMAGES_DIR.iterdir() if d.is_dir()])
print(f"Total categories: {len(categories)}")
print()

all_images = []
for cat in categories:
    imgs = sorted(cat.glob("*.jpg")) + sorted(cat.glob("*.png")) + sorted(cat.glob("*.webp"))
    all_images.extend(imgs)
    print(f"  {cat.name:25s}  {len(imgs):3d} images")

print(f"\n  {'TOTAL':25s}  {len(all_images):3d} images")

In [ ]:
# Show a random sample of 8 images
random.seed(42)
sample = random.sample(all_images, min(8, len(all_images)))
show_image_grid(sample, ncols=4, title="Random Sample from Synthetic PPE Dataset")

**Think about it:** Look at the scene categories above. Which types do you think
will be hardest for an automated detector? Why?

Consider: camera angle, distance to workers, lighting, occlusion.

---
## 3. Zero-Shot Baseline with YOLOe

Before training anything, let's see what an **open-vocabulary** detector can do
out of the box. YOLOe can detect any concept you describe in text — no training needed.

We'll run it with `["hard hat", "person"]` and evaluate against the validation set.

**Questions to think about:**
- What mAP50 do you expect from a model that has never seen construction sites?
- What's the inference speed? Is this production-viable at 30+ FPS?

In [ ]:
# Run the zero-shot baseline evaluation
# This script loads YOLOe-26n-seg, runs inference on the val set,
# and computes mAP50, per-class AP, and inference speed.
# ~2 minutes on A40 GPU, ~5 minutes on CPU

import subprocess, sys

subprocess.run(
    [sys.executable, str(DATA_DIR / "evaluate_yoloe_26n.py")],
    capture_output=False,
    check=True,
)

# You should see:
#   mAP50:     ~0.45-0.55
#   AP50 hardhat:  lower (the harder class)
#   AP50 person:   higher (easier to detect)
#   Avg inference:  ~50-150 ms/img on CPU


In [ ]:
# Visualize zero-shot detections on images from different categories
from ultralytics import YOLO

model = YOLO("yoloe-26n-seg.pt")
model.set_classes(["hard hat", "person"])

# Pick one image from easy, cctv, and ambiguous
test_images = [
    IMAGES_DIR / "easy" / "easy_01.jpg",
    IMAGES_DIR / "cctv" / "cctv_01.jpg",
    IMAGES_DIR / "ambiguous" / "ambiguous_01.jpg",
]

for img_path in test_images:
    if img_path.exists():
        results = model(str(img_path), device=DEVICE, verbose=False)
        results[0].show()
        n_det = len(results[0].boxes) if results[0].boxes is not None else 0
        print(f"{img_path.parent.name}/{img_path.name}: {n_det} detections")
    else:
        # Try png/webp if jpg not found
        for ext in [".png", ".webp"]:
            alt = img_path.with_suffix(ext)
            if alt.exists():
                results = model(str(alt), device=DEVICE, verbose=False)
                results[0].show()
                n_det = len(results[0].boxes) if results[0].boxes is not None else 0
                print(f"{alt.parent.name}/{alt.name}: {n_det} detections")
                break


### Baseline Observations

Write your observations here:
- mAP50: ___
- Inference speed: ___ ms/image
- Which class does the model struggle with most? ___
- Are there scene types where it fails completely? ___

---
## 4. Prompt Experimentation

The zero-shot model's performance depends heavily on **how you describe what to find**.
Before committing to a labeling run, let's experiment with different prompts.

We'll run the same detector on a few sample images with different text descriptions
and compare how many objects each prompt finds.

In [ ]:
# ── Prompt Experiment ────────────────────────────────────────────────────────
# Compare detection counts across different prompt formulations.
# We test 3 prompts on 5 sample images and count detections per class.

from ultralytics import YOLO

# Pick 5 diverse images for the experiment
prompt_test_images = []
for cat_name in ["easy", "cctv", "mixed_compliance", "close_up", "highway"]:
    cat_dir = IMAGES_DIR / cat_name
    if cat_dir.exists():
        imgs = sorted(cat_dir.glob("*.jpg")) + sorted(cat_dir.glob("*.png")) + sorted(cat_dir.glob("*.webp"))
        if imgs:
            prompt_test_images.append(imgs[0])

print(f"Testing on {len(prompt_test_images)} images:")
for p in prompt_test_images:
    print(f"  {p.parent.name}/{p.name}")

# Define prompts to compare
# Each entry: (prompt_label, [class_list])
prompts_to_test = [
    ("hard hat, person",                ["hard hat", "person"]),
    ("helmet, person",                  ["helmet", "person"]),
    ("safety helmet, construction worker", ["safety helmet", "construction worker"]),
]

# Run experiment
print("\n" + "=" * 70)
print(f"{'Prompt':<45} {'Class 0':>10} {'Class 1':>10} {'Total':>8}")
print("=" * 70)

for prompt_label, class_list in prompts_to_test:
    model_exp = YOLO("yoloe-26n-seg.pt")
    model_exp.set_classes(class_list)

    total_c0 = 0
    total_c1 = 0

    for img_path in prompt_test_images:
        results = model_exp.predict(str(img_path), conf=0.25, device=DEVICE, verbose=False)
        if results and results[0].boxes is not None:
            classes = results[0].boxes.cls.tolist()
            total_c0 += classes.count(0)
            total_c1 += classes.count(1)

    total = total_c0 + total_c1
    print(f"{prompt_label:<45} {total_c0:>10} {total_c1:>10} {total:>8}")

    del model_exp

print("=" * 70)
print(f"\nClass 0 = first word in prompt (hat/helmet), Class 1 = second (person/worker)")


**What do you notice?** Which prompt found the most objects? The fewest?

Try 2 more prompts of your own in the cell below. Some ideas to explore:
- Does adding "construction" context help or hurt?
- What about simpler, shorter words?
- What happens with plural forms?

> **Ask your copilot:** "What other prompts could I try for detecting
> hardhats in construction site images? Give me 5 variations and explain
> why each might work differently."

In [ ]:
# ── Your Own Prompts ─────────────────────────────────────────────────────────
# Add your own prompt experiments below.
# Change the class_list and re-run to compare.

my_prompts = [
    # Add your experiments here, e.g.:
    # ("yellow helmet, worker",  ["yellow helmet", "worker"]),
    # ("protective headgear, person",  ["protective headgear", "person"]),
]

if my_prompts:
    print(f"{'Prompt':<45} {'Class 0':>10} {'Class 1':>10} {'Total':>8}")
    print("=" * 70)

    for prompt_label, class_list in my_prompts:
        model_exp = YOLO("yoloe-26n-seg.pt")
        model_exp.set_classes(class_list)

        total_c0 = 0
        total_c1 = 0

        for img_path in prompt_test_images:
            results = model_exp.predict(str(img_path), conf=0.25, device=DEVICE, verbose=False)
            if results and results[0].boxes is not None:
                classes = results[0].boxes.cls.tolist()
                total_c0 += classes.count(0)
                total_c1 += classes.count(1)

        total = total_c0 + total_c1
        print(f"{prompt_label:<45} {total_c0:>10} {total_c1:>10} {total:>8}")
        del model_exp

    print("=" * 70)
else:
    print("Add your prompt experiments to the my_prompts list above and re-run!")


---
## 5. Auto-Labeling with Foundation Models

The zero-shot model gives us a baseline, but it is too slow for production.
To get a **fast, specialized** detector, we need to train one --
and for that, we need labeled data.

Instead of hand-labeling 91 images, we will use a foundation model as an
auto-labeler ("teacher"). It produces YOLO-format bounding box annotations
that we can use to train a smaller, faster model.

### Two Auto-Labeling Options

| Tool | Model | Speed | Access |
|------|-------|-------|--------|
| `auto_label_sam3_hf.py` | SAM3 (Segment Anything 3) | ~2-3 sec/image | **Gated** -- needs HuggingFace approval |
| `auto_label_qwen3_vl.py` | Qwen 3.5 (Vision-Language Model) | ~3-5 sec/image | **Ungated** -- works immediately |

Both produce the same YOLO-format output, so all downstream tools work identically.
Pick whichever is available to you.

**Fast finishers:** Try running BOTH auto-labelers and compare their outputs in Notebook 02.

### Labeling Mode

Each labeler supports a `--mode` flag:
- `exp_a` -- 2-class: **hardhat** + **person** (recommended)
- `3class` -- 3-class: **hardhat** + **no_hardhat** + **person**

Think about what you learned on Day 1 about the "not" paradox.
Which mode makes more sense for this problem? Pick one and run it.

### Prompts Used by the Auto-Labelers

**SAM3 HF** sends **one concept per call** to the foundation model.
Each concept is a separate inference call -- you cannot combine multiple concepts
in one prompt. The labeling script runs two calls per image.

For SAM3 HF in `exp_a` mode, the built-in prompts are:
- **Call 1 → Class 0 (hardhat):** `"hard hat"`
- **Call 2 → Class 1 (person):** `"person"`

**Qwen 3.5** uses a single combined prompt with comma-separated concepts:
- `"hard hat, person"` (one call, multiple concepts parsed from the response)

You can inspect and modify these prompts in the script source code.
After Section 4, you have intuition about which words work best --
consider whether the auto-labeler is using optimal prompts!

> **Ask your copilot:** "What are the tradeoffs between detecting
> 'no_hardhat' as a class versus detecting only 'hardhat' and 'person'
> and figuring out compliance in code?"

In [ ]:
# ── Auto-Labeling ────────────────────────────────────────────────────────────
# Pick ONE of the two options below and run it.
# ~5-8 minutes for 91 images on A40 GPU
#
# PROMPTS USED (exp_a mode):
#   SAM3 HF:  one call per concept -> "hard hat" (class 0), "person" (class 1)
#   Qwen 3.5: comma-separated -> "hard hat, person"
#
# You can change --mode to "3class" if you want to try the 3-class approach.

import subprocess, sys

# ── Option A: SAM3 (if you have HuggingFace approval) ──
subprocess.run([
    sys.executable, str(DATA_DIR / "auto_label_sam3_hf.py"),
    "--mode", "exp_a",
    "--source-dir", str(IMAGES_DIR),
    "--output-dir", str(DATA_DIR / "ppe_dataset_sam3_exp_a"),
    "--device", DEVICE,
], check=True)

# ── Option B: Qwen 3.5 (ungated, no approval needed) ──
# Uncomment the block below and COMMENT OUT Option A above if SAM3 is unavailable.
# The 8B model needs ~16GB VRAM. For less VRAM, add:
#   "--model-id", "Qwen/Qwen3-VL-4B-Instruct"
#
# subprocess.run([
#     sys.executable, str(DATA_DIR / "auto_label_qwen3_vl.py"),
#     "--mode", "exp_a",
#     "--source-dir", str(IMAGES_DIR),
#     "--output-dir", str(DATA_DIR / "ppe_dataset_qwen3vl"),
#     "--device", DEVICE,
# ], check=True)

---
## 6. Quick Label Inspection

**Never train on labels you haven't looked at.**

Let's count what the auto-labeler produced and visualize a few examples.

In [ ]:
# Count labels per class and split
# Adjust the path if you used Qwen 3.5 instead of SAM3
LABELED_DIR = DATA_DIR / "ppe_dataset_sam3_exp_a"  # or "ppe_dataset_qwen3vl"

if LABELED_DIR.exists():
    counts = count_labels(LABELED_DIR)
    for split, split_counts in counts.items():
        print(f"\n{split}:")
        for name, count in sorted(split_counts.items()):
            print(f"  {name:15s}  {count:4d}")
    print()
    total = sum(c for sc in counts.values() for c in sc.values())
    print(f"Total annotations: {total}")
else:
    print(f"Dataset not found at {LABELED_DIR}")
    print("Run the auto-labeling cell above first.")

# You should see hundreds of annotations across train and val.

In [ ]:
# Visualize labels on validation images
import yaml

if LABELED_DIR.exists():
    yaml_path = LABELED_DIR / "dataset.yaml"
    class_names = {}
    if yaml_path.exists():
        with open(yaml_path) as f:
            cfg = yaml.safe_load(f)
        class_names = cfg.get("names", {})

    val_images = sorted((LABELED_DIR / "images" / "val").glob("*.*"))
    print(f"Showing {min(5, len(val_images))} of {len(val_images)} validation images\n")

    for img_path in val_images[:5]:
        label_path = LABELED_DIR / "labels" / "val" / f"{img_path.stem}.txt"
        show_labeled_image(img_path, label_path, class_names=class_names)
else:
    print(f"Dataset not found at {LABELED_DIR} — run auto-labeling first.")

**Look carefully at the labels above:**
- Do you see any obvious labeling errors?
- Are there tiny bounding boxes that look like noise?
- Are any objects missed entirely?
- How tight are the boxes around the actual objects?

---
## 7. Phase 1 Observations

Before moving on, record what you've learned so far.

### Zero-Shot Baseline
- mAP50: ___
- Best class: ___
- Worst class: ___
- Inference speed: ___ ms/image

### Prompt Experiment
- Best prompt: ___
- Worst prompt: ___
- Key insight about prompt choice: ___

### Auto-Labeling
- Which labeler did you use? ___
- Which mode (exp_a / 3class)? ___
- Total labels generated: ___
- Label quality impression (1-5): ___
- Common errors you noticed: ___

> **Ask your copilot:** "Given these auto-labeling results, what should
> I focus on improving before I train a model? What's likely to have the
> biggest impact on final model quality?"

---

**Next**: Open `02_inspect_iterate_train.ipynb` to clean the labels and train a fast detector.